# 10. Prepare Intellytics Source Views

ODS 원천 테이블을 raw mirror view로 복제하고, 기존 분류 결과를 붙인 파이프라인 작업용 view와 최종 Tableau view를 생성합니다.

- 원천 ODS: `kic_data_ods.intellytics_voc.intellytics_display_online_voc`
- 원천 mirror view: `sandbox.t_online_voc_analysis.intellytics_display_online_voc_raw`
- 파이프라인 입력 work view: `sandbox.t_online_voc_analysis.intellytics_display_online_voc_work`
- 최종 Tableau view: `sandbox.t_online_voc_analysis.intellytics_display_online_voc` (ODS 원본 컬럼 + `pred_topic`, `topic_group`)

ODS의 `cate_1_depth`, `cate_2_depth`에 `01.`, `01-01.` 같은 번호 prefix가 포함될 수 있으므로 mirror view에는 `cate_1_depth_rev`, `cate_2_depth_rev` 정규화 컬럼을 추가합니다. work view는 기존 파이프라인/분류 산출물과 맞도록 `cate_1_depth`, `cate_2_depth`를 정규화된 값으로 노출하고, 원본 값은 `raw_cate_1_depth`, `raw_cate_2_depth`로 보존합니다.


In [ ]:
from pyspark.sql import functions as F

PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"

import sys
if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

from common.config_loader import load_config, get_output_table, get_source_table
from ml.final_output_view import create_or_replace_final_classification_view

config = load_config(f"{PROJECT_ROOT}/config/settings_intellytics.yaml")

SOURCE_TABLE = get_source_table(config, "raw_ods_table")
TARGET_SCHEMA = "sandbox.t_online_voc_analysis"
MIRROR_VIEW = get_source_table(config, "raw_mirror_table")
WORK_VIEW = "sandbox.t_online_voc_analysis.intellytics_display_online_voc_work"
FINAL_VIEW = get_source_table(config, "final_tableau_view")

FINAL_DETAIL_TABLE = get_output_table(config, "classification_detail_final")
TOPIC_GROUP_TABLE = get_output_table(config, "topic_group")

# view 이름에 기존 managed table이 있으면 view 생성이 실패할 수 있습니다.
# True면 기존 table을 drop한 뒤 view로 교체합니다.
REPLACE_EXISTING_TABLE_WITH_VIEW = True

print({
    "source_table": SOURCE_TABLE,
    "mirror_view": MIRROR_VIEW,
    "work_view": WORK_VIEW,
    "final_view": FINAL_VIEW,
    "configured_raw_review_table": get_source_table(config, "raw_review_table"),
    "final_detail_table": FINAL_DETAIL_TABLE,
    "topic_group_table": TOPIC_GROUP_TABLE,
    "replace_existing_table_with_view": REPLACE_EXISTING_TABLE_WITH_VIEW,
})


In [ ]:
# 1. 생성 전 원천/대상 상태 확인
source_cnt = spark.table(SOURCE_TABLE).count()
mirror_exists_before = spark.catalog.tableExists(MIRROR_VIEW)
work_exists_before = spark.catalog.tableExists(WORK_VIEW)
final_view_exists_before = spark.catalog.tableExists(FINAL_VIEW)
final_exists = spark.catalog.tableExists(FINAL_DETAIL_TABLE)
topic_group_exists = spark.catalog.tableExists(TOPIC_GROUP_TABLE)

print({
    "source_cnt": source_cnt,
    "mirror_exists_before": mirror_exists_before,
    "work_exists_before": work_exists_before,
    "final_view_exists_before": final_view_exists_before,
    "final_exists": final_exists,
    "topic_group_exists": topic_group_exists,
})

display(spark.sql(f"DESCRIBE TABLE {SOURCE_TABLE}"))


In [ ]:
# 2. sandbox schema 준비 후 mirror/work view 생성
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}")

# ODS cate_1_depth/cate_2_depth에 붙은 numbering prefix를 제거합니다.
# 예: "08. Smart Features & User Experience (UX)" -> "Smart Features & User Experience (UX)"
# 예: "08-06. Remote Control Usability" -> "Remote Control Usability"
# 예: "16. Content Usage Context" -> "Content Usage Context"
mirror_view_sql = f"""
CREATE OR REPLACE VIEW {MIRROR_VIEW} AS
SELECT
  *,
  TRIM(REGEXP_REPLACE(TRIM(CAST(cate_1_depth AS STRING)), '^[0-9]+[.] ', '')) AS cate_1_depth_rev,
  CASE
    WHEN cate_2_depth IS NULL THEN NULL
    ELSE TRIM(
      REGEXP_REPLACE(
        REGEXP_REPLACE(TRIM(CAST(cate_2_depth AS STRING)), '^[0-9]+-[0-9]+[.] ', ''),
        '^[0-9]+[.] ',
        ''
      )
    )
  END AS cate_2_depth_rev
FROM {SOURCE_TABLE}
"""

def create_or_replace_view(view_name: str, sql_text: str) -> None:
    try:
        spark.sql(sql_text)
    except Exception as error:
        print(f"CREATE OR REPLACE VIEW failed: {view_name}. Existing object may be a managed table.")
        print(repr(error))
        if not REPLACE_EXISTING_TABLE_WITH_VIEW:
            raise
        spark.sql(f"DROP TABLE IF EXISTS {view_name}")
        spark.sql(sql_text)

create_or_replace_view(MIRROR_VIEW, mirror_view_sql)
print("created_mirror_view =", MIRROR_VIEW)

# memo_id 생성 시 사용하는 정규화 로직을 SQL로 재현합니다.
memo_norm_sql = "TRIM(REGEXP_REPLACE(REGEXP_REPLACE(LOWER(TRANSLATE(COALESCE(CAST(r.memo AS STRING), ''), '　', ' ')), '[^0-9a-zA-Z가-힣\\s]', ' '), '\\s+', ' '))"
standard_cate_1_sql = "COALESCE(NULLIF(r.cate_1_depth_rev, ''), TRIM(CAST(r.cate_1_depth AS STRING)))"
standard_cate_2_sql = "COALESCE(NULLIF(r.cate_2_depth_rev, ''), TRIM(CAST(r.cate_2_depth AS STRING)))"

# 부가기능 alias 그룹은 최종 분류가 Accessibility Features 기준 memo_id로 저장되므로,
# work view에서 기존 분류를 붙일 때도 alias cate_2_depth 기준 memo_id를 계산합니다.
alias_raw_cate_2_sql = f"""
CASE
  WHEN {standard_cate_1_sql} = 'Smart Features & User Experience (UX)'
   AND {standard_cate_2_sql} IN (
     'Accessibility Features',
     'Ambient & Gallery Mode',
     'Multi View & Screen Split',
     'Program Guide (EPG)',
     'Recording & Utility Features'
   )
  THEN 'Accessibility Features'
  ELSE {standard_cate_2_sql}
END
""".strip()

alias_work_cate_2_sql = """
CASE
  WHEN r.cate_1_depth = 'Smart Features & User Experience (UX)'
   AND r.cate_2_depth IN (
     'Accessibility Features',
     'Ambient & Gallery Mode',
     'Multi View & Screen Split',
     'Program Guide (EPG)',
     'Recording & Utility Features'
   )
  THEN 'Accessibility Features'
  ELSE r.cate_2_depth
END
""".strip()

raw_with_memo_sql = f"""
SELECT
  r.* EXCEPT (cate_1_depth, cate_2_depth),
  r.cate_1_depth AS raw_cate_1_depth,
  r.cate_2_depth AS raw_cate_2_depth,
  {standard_cate_1_sql} AS cate_1_depth,
  {standard_cate_2_sql} AS cate_2_depth,
  SHA2(
    CONCAT_WS(
      '||',
      COALESCE(CAST({standard_cate_1_sql} AS STRING), ''),
      COALESCE(CAST(({alias_raw_cate_2_sql}) AS STRING), ''),
      COALESCE(CAST(CAST(r.sc_measurement AS INT) AS STRING), ''),
      {memo_norm_sql}
    ),
    256
  ) AS _work_memo_id
FROM {MIRROR_VIEW} r
"""

if final_exists and topic_group_exists:
    work_view_sql = f"""
CREATE OR REPLACE VIEW {WORK_VIEW} AS
WITH raw_with_memo AS (
  {raw_with_memo_sql}
), final_latest AS (
  SELECT *
  FROM (
    SELECT
      f.*,
      ROW_NUMBER() OVER (
        PARTITION BY f.cate_1_depth, f.cate_2_depth, f.sc_measurement, f.memo_id
        ORDER BY f.created_at DESC
      ) AS _rn
    FROM {FINAL_DETAIL_TABLE} f
  )
  WHERE _rn = 1
), topic_group_latest AS (
  SELECT *
  FROM (
    SELECT
      g.*,
      ROW_NUMBER() OVER (
        PARTITION BY g.cate_1_depth, g.cate_2_depth, g.sc_measurement, g.topic
        ORDER BY g.created_at DESC
      ) AS _rn
    FROM {TOPIC_GROUP_TABLE} g
  )
  WHERE _rn = 1
)
SELECT
  r.* EXCEPT (_work_memo_id),
  f.memo_id AS existing_memo_id,
  f.pred_topic AS existing_pred_topic,
  f.pred_topic_type AS existing_pred_topic_type,
  f.classification_stage AS existing_classification_stage,
  f.confidence_score AS existing_confidence_score,
  COALESCE(g.topic_group, CASE WHEN f.pred_topic IS NOT NULL THEN '기타' END) AS existing_topic_group,
  f.created_at AS existing_classification_created_at
FROM raw_with_memo r
LEFT JOIN final_latest f
  ON r.cate_1_depth = f.cate_1_depth
 AND ({alias_work_cate_2_sql}) = f.cate_2_depth
 AND CAST(r.sc_measurement AS INT) = CAST(f.sc_measurement AS INT)
 AND r._work_memo_id = f.memo_id
LEFT JOIN topic_group_latest g
  ON f.cate_1_depth = g.cate_1_depth
 AND f.cate_2_depth = g.cate_2_depth
 AND CAST(f.sc_measurement AS INT) = CAST(g.sc_measurement AS INT)
 AND f.pred_topic = g.topic
"""
elif final_exists:
    work_view_sql = f"""
CREATE OR REPLACE VIEW {WORK_VIEW} AS
WITH raw_with_memo AS (
  {raw_with_memo_sql}
), final_latest AS (
  SELECT *
  FROM (
    SELECT
      f.*,
      ROW_NUMBER() OVER (
        PARTITION BY f.cate_1_depth, f.cate_2_depth, f.sc_measurement, f.memo_id
        ORDER BY f.created_at DESC
      ) AS _rn
    FROM {FINAL_DETAIL_TABLE} f
  )
  WHERE _rn = 1
)
SELECT
  r.* EXCEPT (_work_memo_id),
  f.memo_id AS existing_memo_id,
  f.pred_topic AS existing_pred_topic,
  f.pred_topic_type AS existing_pred_topic_type,
  f.classification_stage AS existing_classification_stage,
  f.confidence_score AS existing_confidence_score,
  CASE WHEN f.pred_topic IS NOT NULL THEN '기타' END AS existing_topic_group,
  f.created_at AS existing_classification_created_at
FROM raw_with_memo r
LEFT JOIN final_latest f
  ON r.cate_1_depth = f.cate_1_depth
 AND ({alias_work_cate_2_sql}) = f.cate_2_depth
 AND CAST(r.sc_measurement AS INT) = CAST(f.sc_measurement AS INT)
 AND r._work_memo_id = f.memo_id
"""
else:
    work_view_sql = f"""
CREATE OR REPLACE VIEW {WORK_VIEW} AS
WITH raw_with_memo AS (
  {raw_with_memo_sql}
)
SELECT
  r.* EXCEPT (_work_memo_id),
  CAST(NULL AS STRING) AS existing_memo_id,
  CAST(NULL AS STRING) AS existing_pred_topic,
  CAST(NULL AS STRING) AS existing_pred_topic_type,
  CAST(NULL AS STRING) AS existing_classification_stage,
  CAST(NULL AS DOUBLE) AS existing_confidence_score,
  CAST(NULL AS STRING) AS existing_topic_group,
  CAST(NULL AS TIMESTAMP) AS existing_classification_created_at
FROM raw_with_memo r
"""

create_or_replace_view(WORK_VIEW, work_view_sql)
print("created_work_view =", WORK_VIEW)

# 최종 view는 ODS 전체 행을 유지하며 pred_topic, topic_group 두 컬럼만 추가합니다.
final_view_result = create_or_replace_final_classification_view(
    spark,
    config,
    replace_existing_table_with_view=REPLACE_EXISTING_TABLE_WITH_VIEW,
)
print("created_final_view =", final_view_result)


In [ ]:
# 3. row count / post_no 기준 검증 + category mapping 조인 확인
source_df = spark.table(SOURCE_TABLE)
mirror_df = spark.table(MIRROR_VIEW)
work_df = spark.table(WORK_VIEW)
final_view_df = spark.table(FINAL_VIEW)
category_mapping_table = config["reference"]["category_mapping_table"]

summary_rows = []
for name, df in [("ods_source", source_df), ("mirror_view", mirror_df), ("work_view", work_df), ("final_view", final_view_df)]:
    item = {
        "source": name,
        "row_cnt": df.count(),
    }
    if "post_no" in df.columns:
        item["distinct_post_no_cnt"] = df.select("post_no").dropDuplicates().count()
    if "is_lifestyle" in df.columns:
        item["lifestyle_n_cnt"] = df.where(F.col("is_lifestyle") == "N").count()
        item["lifestyle_y_cnt"] = df.where(F.col("is_lifestyle") == "Y").count()
    if "existing_pred_topic" in df.columns:
        item["existing_classified_row_cnt"] = df.where(F.col("existing_pred_topic").isNotNull()).count()
    summary_rows.append(item)

summary_df = spark.createDataFrame(summary_rows)
display(summary_df)

counts = {row["source"]: row["row_cnt"] for row in summary_df.collect()}
if counts.get("ods_source") != counts.get("mirror_view"):
    raise ValueError(f"ODS/mirror row count mismatch: {counts}")
if counts.get("ods_source") != counts.get("work_view"):
    raise ValueError(f"ODS/work row count mismatch: {counts}")
if counts.get("ods_source") != counts.get("final_view"):
    raise ValueError(f"ODS/final row count mismatch: {counts}")

print("source/work view validation passed")

print("mirror normalized category sample")
display(
    mirror_df
    .select("cate_1_depth", "cate_1_depth_rev", "cate_2_depth", "cate_2_depth_rev")
    .dropDuplicates()
    .orderBy("cate_1_depth", "cate_2_depth")
)

category_join_check_sql = f"""
WITH work_category AS (
  SELECT cate_1_depth, cate_2_depth, COUNT(*) AS row_cnt
  FROM {WORK_VIEW}
  GROUP BY cate_1_depth, cate_2_depth
), mapping_category AS (
  SELECT DISTINCT cate_1_depth, cate_2_depth, cate_1_depth_kor, cate_2_depth_kor
  FROM {category_mapping_table}
)
SELECT
  w.cate_1_depth,
  w.cate_2_depth,
  m.cate_1_depth_kor,
  m.cate_2_depth_kor,
  w.row_cnt,
  CASE WHEN m.cate_1_depth IS NULL THEN 'unmatched' ELSE 'matched' END AS mapping_status
FROM work_category w
LEFT JOIN mapping_category m
  ON w.cate_1_depth = m.cate_1_depth
 AND (
      w.cate_2_depth = m.cate_2_depth
      OR (w.cate_2_depth IS NULL AND m.cate_2_depth IS NULL)
 )
ORDER BY mapping_status DESC, w.cate_1_depth, w.cate_2_depth
"""
category_join_check_df = spark.sql(category_join_check_sql)
display(category_join_check_df)

unmatched_cnt = category_join_check_df.where(F.col("mapping_status") == "unmatched").count()
print({"category_mapping_unmatched_group_cnt": unmatched_cnt})
if unmatched_cnt > 0:
    print("Some normalized source categories are not mapped. Review the displayed unmatched rows before downstream runs.")


In [ ]:
# 4. 파이프라인 설정과 연결 확인
configured_source = get_source_table(config, "raw_review_table")
configured_mirror = get_source_table(config, "raw_mirror_table")
configured_final = get_source_table(config, "final_tableau_view")

print({
    "configured_raw_review_table": configured_source,
    "configured_raw_mirror_table": configured_mirror,
    "configured_final_tableau_view": configured_final,
    "matches_work_view": configured_source == WORK_VIEW,
    "matches_mirror_view": configured_mirror == MIRROR_VIEW,
    "matches_final_view": configured_final == FINAL_VIEW,
})

if configured_source != WORK_VIEW:
    raise ValueError(
        f"settings_intellytics.yaml raw_review_table is {configured_source}, expected {WORK_VIEW}"
    )
if configured_mirror != MIRROR_VIEW:
    raise ValueError(
        f"settings_intellytics.yaml raw_mirror_table is {configured_mirror}, expected {MIRROR_VIEW}"
    )
if configured_final != FINAL_VIEW:
    raise ValueError(
        f"settings_intellytics.yaml final_tableau_view is {configured_final}, expected {FINAL_VIEW}"
    )
